In [0]:
-- Create Silver streaming table for cleaned and validated orders
CREATE OR REFRESH STREAMING TABLE first_data_engineering_project.silver.silver_orders

-- Describe the purpose of the Silver table
COMMENT "Cleaned and validated orders table"

-- Define Silver table properties
TBLPROPERTIES (
    "quality" = "silver",
    "pipelines.reset.allowed" = false
)

-- Define data quality expectations
(
    -- Order ID must exist and be a positive value
    CONSTRAINT valid_order_id
        EXPECT (order_id IS NOT NULL AND order_id > 0)
        ON VIOLATION DROP ROW,

    -- Customer ID must exist and be a positive value
    CONSTRAINT valid_customer_id
        EXPECT (customer_id IS NOT NULL AND customer_id > 0)
        ON VIOLATION DROP ROW,

    -- Store ID must exist and be a positive value
    CONSTRAINT valid_store_id
        EXPECT (store_id IS NOT NULL AND store_id > 0)
        ON VIOLATION DROP ROW,

    -- Order date must exist
    CONSTRAINT valid_order_date
        EXPECT (order_date IS NOT NULL)
        ON VIOLATION DROP ROW
)

AS

-- Deduplicate orders and keep the most recently ingested record
WITH duplicate_orders AS (

    SELECT
        *,

        -- Assign row number 1 to the most recently ingested
        -- record for each order_id
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY ingestion_timestamp DESC
        ) AS row_num

    -- Read orders from the Bronze streaming table
    FROM STREAM first_data_engineering_project.bronze.bronze_orders
)

-- Select the cleaned Silver columns
SELECT
    order_id,
    customer_id,
    store_id,
    order_date,
    promotion_id,
    ingestion_timestamp,
    source_file

-- Read the deduplicated orders
FROM duplicate_orders

-- Keep only the most recently ingested record for each order_id
WHERE row_num = 1;